# Try Sentinel in your browser

Sentinel checks a prompt for injection attacks **before** it reaches your model. This notebook runs the real code: the rule shield, the keyword layer and, optionally, a trained classifier. Nothing is sent anywhere; your text stays in this Colab session.

1. Run **Install** (about 2-4 minutes the first time).
2. Run **Load**.
3. Try the examples, then your own text.

Honest numbers on our 55-prompt blind set: rules only catch ~10% of attacks, with a classifier ~73-93% (and 4-12% false blocks). See the [ai-sentinel README](https://github.com/yurtseveronder-design/ai-sentinel#readme).

In [ ]:
# Install (pinned to exact commits, so this notebook keeps working)
AI_SENTINEL_REF = "b4c13f72e4bacf7fa3e32f0e7bf731b4b0f19da4"
SEMANTIC_REF = "891b47701c117afa67454bd7f01e5b1fe9af5e65"
DEMO_REF = "main"
!pip install -q "ai-sentinel @ git+https://github.com/yurtseveronder-design/ai-sentinel.git@{AI_SENTINEL_REF}" pyyaml
!pip install -q "sentinel-semantic[classifier] @ git+https://github.com/yurtseveronder-design/sentinel-semantic.git@{SEMANTIC_REF}"
!rm -rf sentinel-rag-demo && git clone -q --depth 1 -b {DEMO_REF} https://github.com/yurtseveronder-design/sentinel-rag-demo.git
import sys; sys.path.insert(0, "sentinel-rag-demo")
print("installed")

In [ ]:
# Load: rules always; the classifier modes download a ~700 MB model each (first use only)
from playground.engine import Playground, MODES
from playground.examples import EXAMPLES
pg = Playground.create()
for m in pg.modes():
    print(("OK      " if m["available"] else "missing "), m["label"])

In [ ]:
def show(text, mode="rules"):
    r = pg.check(text, mode)
    print(("BLOCK" if r["action"] == "BLOCK" else "ALLOW"), "-", MODES[mode]["label"])
    if r["caught_by"]: print("  caught by:", ", ".join(r["caught_by"]))
    for why in r["rules"]["reasons"]: print("  rules:", why)
    s = r["semantic"]
    print("  intent:", s["intent"], "| classifier injection probability:", s["classifier_injection_probability"])

for ex in EXAMPLES:
    print("\n##", ex["title"], "->", ex["text"])
    show(ex["text"], "rules")

## Same examples with a trained classifier
Compare the reworded and Turkish examples, which slip past the rules, and see whether the classifier catches them (no classifier is perfect; check the harmless examples too). Switch `mode` to `"protectai"` if you prefer fewer false alarms.

In [ ]:
mode = "deepset"   # or "protectai" or "rules"
for ex in EXAMPLES:
    print("\n##", ex["title"], "->", ex["text"])
    show(ex["text"], mode)

## Your own text

In [ ]:
text = "Type or paste a prompt here"
mode = "deepset"   # "rules", "protectai" or "deepset"
show(text, mode)